# REMNET Stage 3 — Classification Paired Statistical Analysis
**Direct numerical input edition · same structure as the Stage 2 paired t-test reference notebook.**

No dataset location, CSV input, checkpoint path, model loading or internet required. The original full-precision five-seed **D1 (463 images)** and **D2 (4,931 images)** classification results are declared as numerical Python lists. Seeds are aligned **[7, 49, 112, 345, 476]**.

Descriptive results use **mean ± sample SD (ddof=1)**. The primary D2 family contains six two-sided paired t-tests: REMNET versus three baselines for **accuracy and macro-F1**, with Holm correction across those six tests. Macro-precision and macro-recall are analyzed as a separate exploratory six-test family. 95% Student-t CIs (df=4), Cohen's paired dz, seed-wise differences and exact sign-flip sensitivity p-values are displayed. No superiority claim is made solely from p-values (n=5).

In [3]:
# Stage 3 statistical analysis; all five-seed inputs are literal numbers below.
# No paths, file loading, checkpoint loading, GPUs or internet are necessary.
import itertools
import numpy as np
import pandas as pd
from scipy import stats

SEEDS = [7, 49, 112, 345, 476]
MODELS = ['REMNET', 'ResNet50', 'MobileNetV2', 'EfficientNet-Lite0']
BASELINES = ['ResNet50', 'MobileNetV2', 'EfficientNet-Lite0']
ALPHA = 0.05
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 230)
pd.set_option('display.max_columns', 32)
pd.set_option('display.float_format', lambda x: f'{x:.12f}')

def holm_adjust(p_values):
    p = np.asarray(p_values, dtype=float)
    assert np.isfinite(p).all() and ((0 <= p) & (p <= 1)).all()
    ranks = np.argsort(p, kind='stable')
    out = np.empty(len(p), dtype=float)
    current = 0.0
    for rank, idx in enumerate(ranks):
        current = max(current, min(1.0, (len(p)-rank)*p[idx]))
        out[idx] = current
    return out

def paired_test(remnet, baseline):
    a, b = np.asarray(remnet, dtype=float), np.asarray(baseline, dtype=float)
    assert len(a) == len(b) == 5 and np.isfinite(a).all() and np.isfinite(b).all()
    d = a-b  # REMNET minus CNN; negative energy difference means lower recorded energy
    mean, sd = float(d.mean()), float(d.std(ddof=1))
    if sd == 0:
        raise ValueError('Paired differences have zero sample SD; t-statistic / dz undefined')
    t = mean/(sd/np.sqrt(5))
    p = float(2*stats.t.sf(abs(t), df=4))
    half_width = float(stats.t.ppf(0.975,df=4) * sd/np.sqrt(5))
    # Exact sign-flip randomization sensitivity test: only 32 possible flips at n=5.
    flips = [np.mean(d*np.array(s)) for s in itertools.product((-1,1),repeat=5)]
    exact_p = float(np.mean(np.abs(flips) >= abs(mean)-1e-14))
    return dict(mean_difference=mean, sample_SD_difference=sd,
      CI95_lower=mean-half_width, CI95_upper=mean+half_width,
      Cohen_dz=mean/sd, paired_t=t, df=4,
      p_two_sided=p, exact_sign_flip_p_two_sided=exact_p)

def report(df, cols=None, digits=9):
    print((df if cols is None else df[cols]).to_string(index=False,
       float_format=lambda x:f'{x:.{digits}f}'))


## 1. Original five-seed measurements — values embedded directly in code

In [4]:
# Metric values are percentages. Each list corresponds in order to SEEDS.
classification_values = {'D1': {'REMNET': {'accuracy': [94.6004319654, 94.8164146868, 93.7365010799, 94.1684665227, 93.3045356371],
                   'macro_precision': [94.6181563724,
                                       94.8338353105,
                                       93.5746492541,
                                       94.5726494876,
                                       93.507016715],
                   'macro_recall': [93.3759125241,
                                    93.4488365942,
                                    91.9698058148,
                                    92.4308189454,
                                    91.4288665475],
                   'macro_f1': [93.7913961276, 93.9636710546, 92.4940942008, 93.1294168986, 92.0023203797]},
        'ResNet50': {'accuracy': [90.4967602592,
                                  92.0086393089,
                                  93.9524838013,
                                  90.0647948164,
                                  91.5766738661],
                     'macro_precision': [90.3510351129,
                                         92.2721493901,
                                         93.5497363453,
                                         89.8398123403,
                                         91.677918006],
                     'macro_recall': [89.9789207852,
                                      90.6339778089,
                                      93.3886041679,
                                      88.7591888266,
                                      89.208093288],
                     'macro_f1': [89.8109998638,
                                  91.1763004168,
                                  93.4578801144,
                                  89.0000212123,
                                  89.5541828505]},
        'MobileNetV2': {'accuracy': [90.9287257019,
                                     90.9287257019,
                                     90.0647948164,
                                     89.848812095,
                                     91.1447084233],
                        'macro_precision': [91.900203513,
                                            90.781480856,
                                            91.7567835925,
                                            90.9072147106,
                                            92.2053810994],
                        'macro_recall': [88.6450697959,
                                         89.7542245334,
                                         87.5865975324,
                                         87.7150583754,
                                         89.0262421653],
                        'macro_f1': [89.3617203452,
                                     90.1644315014,
                                     88.4871125146,
                                     88.6001370824,
                                     89.8809676401]},
        'EfficientNet-Lite0': {'accuracy': [90.2807775378,
                                            87.4730021598,
                                            90.7127429806,
                                            90.4967602592,
                                            90.4967602592],
                               'macro_precision': [90.5138290485,
                                                   87.5412244858,
                                                   90.8765504493,
                                                   90.9418445927,
                                                   91.0791525246],
                               'macro_recall': [88.7259898565,
                                                85.2303613632,
                                                88.4714586848,
                                                88.0354166182,
                                                88.6662429238],
                               'macro_f1': [89.3825996851,
                                            85.6629376817,
                                            89.0031803678,
                                            88.6153323416,
                                            89.3373558827]}},
 'D2': {'REMNET': {'accuracy': [63.4962482255, 62.4619752586, 62.7256134658, 62.7053336037, 61.0626647739],
                   'macro_precision': [59.0055723473,
                                       58.7043880574,
                                       58.2857814013,
                                       58.9068766674,
                                       56.7622831534],
                   'macro_recall': [61.9662163713,
                                    61.4836072548,
                                    61.3412141286,
                                    61.1392862929,
                                    60.2746822513],
                   'macro_f1': [59.3615794635, 58.0797418996, 58.7502126577, 59.022538071, 57.2270176913]},
        'ResNet50': {'accuracy': [61.4074224295,
                                  63.1312107078,
                                  59.7241938755,
                                  62.8472926384,
                                  59.5822348408],
                     'macro_precision': [59.8417959797,
                                         60.8081234126,
                                         56.4684364115,
                                         60.9596182349,
                                         57.8612193],
                     'macro_recall': [58.3722601292,
                                      62.4963927304,
                                      59.1166863326,
                                      59.4076087304,
                                      58.6572383549],
                     'macro_f1': [57.8820253598,
                                  59.7027283628,
                                  54.5837106807,
                                  58.5456908385,
                                  55.9382460129]},
        'MobileNetV2': {'accuracy': [51.247211519,
                                     53.96471304,
                                     58.3857229771,
                                     57.8584465626,
                                     55.3640235246],
                        'macro_precision': [48.4425288462,
                                            51.1568020072,
                                            53.9824600434,
                                            55.0703094561,
                                            53.7223062104],
                        'macro_recall': [52.529091192,
                                         54.6704398353,
                                         57.2749849073,
                                         58.6445219237,
                                         55.5297166243],
                        'macro_f1': [47.9205446913,
                                     51.0491831582,
                                     54.6381158603,
                                     55.3752975349,
                                     53.1165588145]},
        'EfficientNet-Lite0': {'accuracy': [57.6759278037,
                                            58.8927195295,
                                            60.2109105658,
                                            55.9115798013,
                                            57.8584465626],
                               'macro_precision': [54.0679091467,
                                                   55.4927370306,
                                                   55.8318359964,
                                                   51.9509133738,
                                                   53.8166957142],
                               'macro_recall': [56.8624324015,
                                                57.9969545228,
                                                58.948933851,
                                                55.7450153095,
                                                55.7827824192],
                               'macro_f1': [53.6298254893,
                                            55.7097518962,
                                            56.1718010888,
                                            52.5093815533,
                                            53.7986725974]}}}

assert all(len(classification_values[ds][m][metric])==5 for ds in ["D1","D2"] for m in MODELS for metric in ["accuracy","macro_precision","macro_recall","macro_f1"])
print("Numerical classification data embedded for 4 models × 5 seeds × D1/D2 × 4 metrics.")

Numerical classification data embedded for 4 models × 5 seeds × D1/D2 × 4 metrics.


## 2. Raw five-seed data, mean ± sample SD, D2 − D1 change

In [5]:
METRICS = ['accuracy','macro_precision','macro_recall','macro_f1']
raw_rows=[]
for dataset in ['D1','D2']:
    for model in MODELS:
        for i, seed in enumerate(SEEDS):
            raw_rows.append({'dataset':dataset,'model':model,'seed':seed,
                             'n_images':463 if dataset=='D1' else 4931,
                             **{metric: classification_values[dataset][model][metric][i] for metric in METRICS}})
raw_classification = pd.DataFrame(raw_rows)
assert len(raw_classification)==40
summary_rows=[]
for dataset in ['D1','D2']:
    for model in MODELS:
        for metric in METRICS:
            v=np.asarray(classification_values[dataset][model][metric],dtype=float)
            summary_rows.append(dict(dataset=dataset,model=model,metric=metric,
                        mean_pct=float(v.mean()),sample_SD_pct=float(v.std(ddof=1))))
classification_summary = pd.DataFrame(summary_rows)
print('FIVE-SEED CLASSIFICATION MEAN AND SAMPLE SD (percentage units):')
report(classification_summary, digits=6)

change_rows=[]
for model in MODELS:
    for i,seed in enumerate(SEEDS):
        change_rows.append({'model':model,'seed':seed,**{
            metric+'_D2_minus_D1_pp':classification_values['D2'][model][metric][i]-classification_values['D1'][model][metric][i]
            for metric in METRICS}})
classification_D2_minus_D1 = pd.DataFrame(change_rows)
print('\nD2 minus D1 accuracy changes (percentage points), matched by seed:')
report(classification_D2_minus_D1[['model','seed','accuracy_D2_minus_D1_pp']],digits=6)


FIVE-SEED CLASSIFICATION MEAN AND SAMPLE SD (percentage units):
dataset              model          metric  mean_pct  sample_SD_pct
     D1             REMNET        accuracy 94.125270       0.618480
     D1             REMNET macro_precision 94.221261       0.629384
     D1             REMNET    macro_recall 92.530848       0.879769
     D1             REMNET        macro_f1 93.076180       0.835764
     D1           ResNet50        accuracy 91.619870       1.522640
     D1           ResNet50 macro_precision 91.538130       1.491506
     D1           ResNet50    macro_recall 90.393757       1.821848
     D1           ResNet50        macro_f1 90.599877       1.787162
     D1        MobileNetV2        accuracy 90.583153       0.583553
     D1        MobileNetV2 macro_precision 91.510213       0.630636
     D1        MobileNetV2    macro_recall 88.545438       0.909831
     D1        MobileNetV2        macro_f1 89.298874       0.748192
     D1 EfficientNet-Lite0        accuracy 89.892009

## 3. Paired D2 comparisons: REMNET versus each standalone CNN

In [6]:
difference_rows=[]
result_rows=[]
for metric in METRICS:
    a=np.asarray(classification_values['D2']['REMNET'][metric],dtype=float)
    for baseline in BASELINES:
        b=np.asarray(classification_values['D2'][baseline][metric],dtype=float)
        for seed,rv,bv in zip(SEEDS,a,b):
            difference_rows.append(dict(metric=metric,seed=seed,baseline=baseline,
               REMNET_pct=float(rv),baseline_pct=float(bv),REMNET_minus_baseline_pp=float(rv-bv)))
        result_rows.append(dict(metric=metric,baseline=baseline,
             REMNET_mean_pct=float(a.mean()),REMNET_sample_SD_pct=float(a.std(ddof=1)),
             baseline_mean_pct=float(b.mean()),baseline_sample_SD_pct=float(b.std(ddof=1)),
             **paired_test(a,b)))
classification_seed_differences=pd.DataFrame(difference_rows)
classification_tests=pd.DataFrame(result_rows)
assert len(classification_seed_differences)==60 and len(classification_tests)==12
classification_tests['family']=np.where(classification_tests.metric.isin(['accuracy','macro_f1']),
   'PRIMARY_6_accuracy_macroF1','EXPLORATORY_6_precision_recall')
classification_tests['Holm_p_within_family']=np.nan
for family in classification_tests.family.unique():
    mask=classification_tests.family==family
    classification_tests.loc[mask,'Holm_p_within_family']=holm_adjust(classification_tests.loc[mask,'p_two_sided'].to_numpy())
classification_tests['Holm_p_all_12_sensitivity']=holm_adjust(classification_tests.p_two_sided.to_numpy())
classification_tests['significant_Holm_0_05']=classification_tests.Holm_p_within_family<ALPHA

print('D2 REMNET-MINUS-CNN DIFFERENCES, EACH SEED (percentage points):')
report(classification_seed_differences,digits=6)
print('\nPRIMARY PAIRED INFERENCE (six Holm-adjusted accuracy/macro-F1 tests):')
report(classification_tests[classification_tests.family.str.startswith('PRIMARY')][
 ['metric','baseline','mean_difference','CI95_lower','CI95_upper','Cohen_dz','p_two_sided','Holm_p_within_family','exact_sign_flip_p_two_sided']],digits=9)
print('\nSECONDARY paired inference (macro-precision and macro-recall):')
report(classification_tests[~classification_tests.family.str.startswith('PRIMARY')][
 ['metric','baseline','mean_difference','CI95_lower','CI95_upper','Cohen_dz','p_two_sided','Holm_p_within_family']],digits=9)
print('\nInterpretation: n=5 yields wide uncertainty; exact sign-flip p cannot be smaller than 0.0625 two-sided.')
print('All four datasets / summaries / per-seed differences / statistical tests are in variables displayed above.')


D2 REMNET-MINUS-CNN DIFFERENCES, EACH SEED (percentage points):
         metric  seed           baseline  REMNET_pct  baseline_pct  REMNET_minus_baseline_pp
       accuracy     7           ResNet50   63.496248     61.407422                  2.088826
       accuracy    49           ResNet50   62.461975     63.131211                 -0.669235
       accuracy   112           ResNet50   62.725613     59.724194                  3.001420
       accuracy   345           ResNet50   62.705334     62.847293                 -0.141959
       accuracy   476           ResNet50   61.062665     59.582235                  1.480430
       accuracy     7        MobileNetV2   63.496248     51.247212                 12.249037
       accuracy    49        MobileNetV2   62.461975     53.964713                  8.497262
       accuracy   112        MobileNetV2   62.725613     58.385723                  4.339890
       accuracy   345        MobileNetV2   62.705334     57.858447                  4.846887
      